In [1]:
from PIL import Image
import numpy as np
sample_path = "/Users/krishna/University/Sem2/Phonetics_Lab/Code/inebriation-voice-detector/data/processed/TRAIN/SOBER/0082006001_h_00.wav_0_0.jpg"

img = Image.open(sample_path)
print("Image size:", img.size)  # (width, height)

# Convert to array to check shape
img_array = np.array(img)
print("Array shape:", img_array.shape)  # (height, width, channels)

Image size: (224, 224)
Array shape: (224, 224, 3)


import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as transforms
import torchvision.models as models
from PIL import Image
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt
from tqdm import tqdm

# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Set device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Define paths to data directories
DATA_ROOT = "path/to/your/data"  # Change this to your actual data path
TRAIN_DIR = os.path.join(DATA_ROOT, "TRAIN")
VAL_DIR = os.path.join(DATA_ROOT, "VALIDATION")
TEST_DIR = os.path.join(DATA_ROOT, "TEST")

# Define classes and class indices
classes = ['SOBER', 'DRUNK']  # SOBER = 0, DRUNK = 1
class_to_idx = {cls: idx for idx, cls in enumerate(classes)}

# Define image transformations
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # ImageNet stats
])

# Create custom dataset class
class SpectrogramDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.samples = []
        
        # Load all file paths and labels
        for class_name in os.listdir(root_dir):
            class_path = os.path.join(root_dir, class_name)
            if os.path.isdir(class_path):
                class_idx = class_to_idx[class_name]
                for filename in os.listdir(class_path):
                    if filename.endswith('.jpg') or filename.endswith('.png'):
                        self.samples.append((os.path.join(class_path, filename), class_idx))
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
        
        return image, label

# Create datasets
train_dataset = SpectrogramDataset(TRAIN_DIR, transform=transform)
val_dataset = SpectrogramDataset(VAL_DIR, transform=transform)
test_dataset = SpectrogramDataset(TEST_DIR, transform=transform)

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Test samples: {len(test_dataset)}")

# Calculate class weights for balanced sampling
def get_class_distribution(dataset):
    count_dict = {0: 0, 1: 0}
    for _, label in dataset:
        count_dict[label] += 1
    return count_dict

class_distribution = get_class_distribution(train_dataset)
print(f"Class distribution in training set: {class_distribution}")

# Compute inverse frequencies for each class and use them as weights
num_samples = len(train_dataset)
class_weights = [num_samples / class_distribution[i] for i in range(len(classes))]
sample_weights = [class_weights[label] for _, label in train_dataset]

# Create weighted sampler for balanced training
sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(train_dataset),
    replacement=True
)

# Create data loaders
BATCH_SIZE = 100

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=sampler,
    num_workers=4,
    pin_memory=True if torch.cuda.is_available() else False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True if torch.cuda.is_available() else False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True if torch.cuda.is_available() else False
)

# Create ResNet-18 model modified for binary classification
def create_model():
    # Load pretrained ResNet-18
    model = models.resnet18(pretrained=True)
    
    # Freeze all layers except final ones
    for param in model.parameters():
        param.requires_grad = False
    
    # Modify the final fully connected layer for binary classification
    num_ftrs = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Linear(num_ftrs, 1),
        nn.Sigmoid()
    )
    
    return model

# Create model
model = create_model()
model = model.to(device)

# Define weighted binary cross-entropy loss
class WeightedBinaryCrossEntropyLoss(nn.Module):
    def __init__(self, weight_pos=0.9, weight_neg=0.1):
        super(WeightedBinaryCrossEntropyLoss, self).__init__()
        self.weight_pos = weight_pos
        self.weight_neg = weight_neg
        
    def forward(self, pred, target):
        target = target.float()
        loss = self.weight_pos * target * torch.log(pred + 1e-7) + \
               self.weight_neg * (1 - target) * torch.log(1 - pred + 1e-7)
        return -torch.mean(loss)

# Set up loss function, optimizer, and learning rate scheduler
criterion = WeightedBinaryCrossEntropyLoss(weight_pos=0.9, weight_neg=0.1)
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

# Training function
def train_model(model, criterion, optimizer, scheduler, train_loader, val_loader, num_epochs=30):
    history = {
        'train_loss': [],
        'val_loss': [],
        'val_accuracy': [],
        'val_precision': [],
        'val_recall': [],
        'val_f1': []
    }
    
    best_val_f1 = 0.0
    
    for epoch in range(num_epochs):
        # Training phase
        model.train()
        running_loss = 0.0
        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
        
        for inputs, labels in progress_bar:
            inputs = inputs.to(device)
            labels = labels.to(device)
            
            # Zero the parameter gradients
            optimizer.zero_grad()
            
            # Forward pass
            outputs = model(inputs)
            outputs = outputs.squeeze()
            loss = criterion(outputs, labels)
            
            # Backward pass and optimize
            loss.backward()
            optimizer.step()
            
            # Track statistics
            running_loss += loss.item() * inputs.size(0)
            progress_bar.set_postfix({'loss': loss.item()})
        
        epoch_loss = running_loss / len(train_loader.dataset)
        history['train_loss'].append(epoch_loss)
        
        # Validation phase
        val_metrics = evaluate(model, val_loader, criterion)
        history['val_loss'].append(val_metrics['loss'])
        history['val_accuracy'].append(val_metrics['accuracy'])
        history['val_precision'].append(val_metrics['precision'])
        history['val_recall'].append(val_metrics['recall'])
        history['val_f1'].append(val_metrics['f1'])
        
        # Step the scheduler
        scheduler.step()
        
        # Print epoch results
        print(f"Epoch {epoch+1}/{num_epochs}:")
        print(f"  Train Loss: {epoch_loss:.4f}")
        print(f"  Val Loss: {val_metrics['loss']:.4f}, Val Accuracy: {val_metrics['accuracy']:.4f}")
        print(f"  Val Precision: {val_metrics['precision']:.4f}, Val Recall: {val_metrics['recall']:.4f}, Val F1: {val_metrics['f1']:.4f}")
        
        # Save best model
        if val_metrics['f1'] > best_val_f1:
            best_val_f1 = val_metrics['f1']
            torch.save(model.state_dict(), 'best_model.pth')
            print("  Saved new best model")
    
    return model, history

# Evaluation function
def evaluate(model, data_loader, criterion):
    model.eval()
    all_preds = []
    all_labels = []
    running_loss = 0.0
    
    with torch.no_grad():
        for inputs, labels in data_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            
            outputs = model(inputs)
            outputs = outputs.squeeze()
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * inputs.size(0)
            
            # Convert probabilities to binary predictions
            preds = (outputs >= 0.5).float()
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    # Calculate metrics
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    
    metrics = {
        'loss': running_loss / len(data_loader.dataset),
        'accuracy': accuracy_score(all_labels, all_preds),
        'precision': precision_score(all_labels, all_preds, zero_division=0),
        'recall': recall_score(all_labels, all_preds, zero_division=0),
        'f1': f1_score(all_labels, all_preds, zero_division=0),
        'confusion_matrix': confusion_matrix(all_labels, all_preds)
    }
    
    return metrics

# Plot training history
def plot_history(history):
    # Plot training & validation loss
    plt.figure(figsize=(12, 4))
    
    plt.subplot(1, 2, 1)
    plt.plot(history['train_loss'], label='Train Loss')
    plt.plot(history['val_loss'], label='Val Loss')
    plt.title('Training and Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    
    plt.subplot(1, 2, 2)
    plt.plot(history['val_accuracy'], label='Accuracy')
    plt.plot(history['val_precision'], label='Precision')
    plt.plot(history['val_recall'], label='Recall')
    plt.plot(history['val_f1'], label='F1 Score')
    plt.title('Validation Metrics')
    plt.xlabel('Epoch')
    plt.ylabel('Score')
    plt.legend()
    
    plt.tight_layout()
    plt.savefig('training_history.png')
    plt.show()

# Main training loop
model, history = train_model(
    model=model,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    train_loader=train_loader,
    val_loader=val_loader,
    num_epochs=30
)

# Plot training history
plot_history(history)

# Load best model for testing
model.load_state_dict(torch.load('best_model.pth'))

# Test the model
test_metrics = evaluate(model, test_loader, criterion)
print("\nTest Results:")
print(f"  Loss: {test_metrics['loss']:.4f}")
print(f"  Accuracy: {test_metrics['accuracy']:.4f}")
print(f"  Precision: {test_metrics['precision']:.4f}")
print(f"  Recall: {test_metrics['recall']:.4f}")
print(f"  F1 Score: {test_metrics['f1']:.4f}")
print("  Confusion Matrix:")
print(test_metrics['confusion_matrix'])

In [2]:

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as transforms
import torchvision.models as models
from PIL import Image
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import time
import random
from pathlib import Path
import cv2

In [3]:
# Set random seed for reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"Random seed set to {seed}")

set_seed(42)

# Set device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Define paths to data directories
DATA_ROOT = "/Users/krishna/University/Sem2/Phonetics_Lab/Code/inebriation-voice-detector/data/processed"  # Change this to your actual data path
TRAIN_DIR = os.path.join(DATA_ROOT, "TRAIN")
VAL_DIR = os.path.join(DATA_ROOT, "VALIDATION")
TEST_DIR = os.path.join(DATA_ROOT, "TEST")

# Create output directory for results
OUTPUT_DIR = "output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Define classes and class indices
classes = ['SOBER', 'DRUNK']  # SOBER = 0, DRUNK = 1
class_to_idx = {cls: idx for idx, cls in enumerate(classes)}
idx_to_class = {idx: cls for idx, cls in enumerate(classes)}

print(f"Class mapping: {class_to_idx}")

Random seed set to 42
Using device: cpu
Class mapping: {'SOBER': 0, 'DRUNK': 1}


In [4]:
# ====================== DATA EXPLORATION ======================

def explore_dataset(data_dir, title):
    """Explore the dataset and visualize class distribution"""
    class_counts = {cls: 0 for cls in classes}
    file_paths = []
    labels = []
    
    # Count samples in each class
    for class_name in classes:
        class_path = os.path.join(data_dir, class_name)
        if os.path.isdir(class_path):
            files = [f for f in os.listdir(class_path) if f.endswith(('.jpg', '.png'))]
            class_counts[class_name] = len(files)
            
            # Store file paths and labels for later use
            for file in files:
                file_paths.append(os.path.join(class_path, file))
                labels.append(class_to_idx[class_name])
    
    # Print statistics
    print(f"\n{title} Dataset Statistics:")
    for cls, count in class_counts.items():
        print(f"  {cls}: {count} samples")
    print(f"  Total: {sum(class_counts.values())} samples")
    
    # Plot class distribution
    plt.figure(figsize=(8, 5))
    sns.barplot(x=list(class_counts.keys()), y=list(class_counts.values()))
    plt.title(f"{title} Dataset Class Distribution")
    plt.xlabel("Class")
    plt.ylabel("Number of Samples")
    for i, count in enumerate(class_counts.values()):
        plt.text(i, count + 5, str(count), ha='center')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f"{title.lower()}_class_distribution.png"))
    plt.close()
    
    return file_paths, labels, class_counts

# Explore all datasets
train_files, train_labels, train_counts = explore_dataset(TRAIN_DIR, "Training")
val_files, val_labels, val_counts = explore_dataset(VAL_DIR, "Validation")
test_files, test_labels, test_counts = explore_dataset(TEST_DIR, "Test")


Training Dataset Statistics:
  SOBER: 5584 samples
  DRUNK: 2458 samples
  Total: 8042 samples

Validation Dataset Statistics:
  SOBER: 4071 samples
  DRUNK: 1688 samples
  Total: 5759 samples

Test Dataset Statistics:
  SOBER: 2348 samples
  DRUNK: 2006 samples
  Total: 4354 samples


In [5]:
# ====================== DATA VISUALIZATION ======================

def visualize_spectrograms(file_paths, labels, num_samples=5, save_path=None):
    """Visualize sample spectrograms from each class"""
    # Get sample images from each class
    samples_by_class = {cls: [] for cls in classes}
    for path, label in zip(file_paths, labels):
        class_name = idx_to_class[label]
        if len(samples_by_class[class_name]) < num_samples:
            samples_by_class[class_name].append(path)
    
    # Create subplot grid
    fig, axes = plt.subplots(len(classes), num_samples, figsize=(15, 5))
    
    for i, class_name in enumerate(classes):
        for j, img_path in enumerate(samples_by_class[class_name]):
            img = cv2.imread(img_path)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            axes[i, j].imshow(img)
            axes[i, j].set_title(f"{class_name}")
            axes[i, j].axis('off')
    
    plt.suptitle("Sample Spectrogram Images")
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path)
    plt.close()

# Visualize sample spectrograms
visualize_spectrograms(
    train_files, 
    train_labels, 
    num_samples=5, 
    save_path=os.path.join(OUTPUT_DIR, "sample_spectrograms.png")
)

In [6]:
# ====================== DATA TRANSFORMS ======================

# Define image transformations
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # ImageNet stats
])

# Visualization transforms (without normalization)
vis_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

def visualize_transforms(file_path, save_path=None):
    """Visualize the effect of transforms on an image"""
    # Load original image
    original_img = Image.open(file_path).convert('RGB')
    
    # Apply transforms
    resized_img = transforms.Resize((224, 224))(original_img)
    tensor_img = transforms.ToTensor()(resized_img)
    normalized_img = transforms.Normalize(
        mean=[0.485, 0.456, 0.406], 
        std=[0.229, 0.224, 0.225]
    )(tensor_img)
    
    # Convert to numpy for visualization
    original_np = np.array(original_img)
    resized_np = np.array(resized_img)
    tensor_np = tensor_img.permute(1, 2, 0).numpy()
    
    # For normalized image, we need to unnormalize for visualization
    normalized_np = normalized_img.permute(1, 2, 0).numpy()
    normalized_np = normalized_np * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
    normalized_np = np.clip(normalized_np, 0, 1)
    
    # Plot
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    
    axes[0].imshow(original_np)
    axes[0].set_title("Original")
    axes[0].axis('off')
    
    axes[1].imshow(resized_np)
    axes[1].set_title("Resized (224x224)")
    axes[1].axis('off')
    
    axes[2].imshow(tensor_np)
    axes[2].set_title("ToTensor [0,1]")
    axes[2].axis('off')
    
    axes[3].imshow(normalized_np)
    axes[3].set_title("Normalized")
    axes[3].axis('off')
    
    plt.suptitle("Transformation Pipeline")
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path)
    plt.close()

# Visualize transforms on a sample image
sample_image = train_files[0]
visualize_transforms(
    sample_image, 
    save_path=os.path.join(OUTPUT_DIR, "transform_visualization.png")
)

In [7]:
# ====================== DATASET CLASS ======================

class SpectrogramDataset(Dataset):
    def __init__(self, root_dir, transform=None, vis_transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.vis_transform = vis_transform  # For visualization
        self.samples = []
        
        # Load all file paths and labels
        for class_name in os.listdir(root_dir):
            class_path = os.path.join(root_dir, class_name)
            if os.path.isdir(class_path):
                class_idx = class_to_idx[class_name]
                for filename in os.listdir(class_path):
                    if filename.endswith('.jpg') or filename.endswith('.png'):
                        self.samples.append((os.path.join(class_path, filename), class_idx))
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert('RGB')
        
        transformed_img = None
        if self.transform:
            transformed_img = self.transform(image)
        
        # For visualization purposes
        vis_img = None
        if self.vis_transform:
            vis_img = self.vis_transform(image)
            return transformed_img, label, vis_img, img_path
        
        return transformed_img, label

# Create datasets
train_dataset = SpectrogramDataset(TRAIN_DIR, transform=transform)
val_dataset = SpectrogramDataset(VAL_DIR, transform=transform)
test_dataset = SpectrogramDataset(TEST_DIR, transform=transform)

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Test samples: {len(test_dataset)}")

Training samples: 8042
Validation samples: 5759
Test samples: 4354


In [8]:
# ====================== BALANCED SAMPLING ======================

def calculate_sampling_weights(dataset):
    """Calculate sampling weights for balanced sampling"""
    # Count samples in each class
    class_counts = {i: 0 for i in range(len(classes))}
    for _, label in dataset:
        class_counts[label] += 1
    
    print("\nClass distribution before balancing:")
    for i, count in class_counts.items():
        print(f"  {idx_to_class[i]}: {count} samples")
    
    # Calculate class weights (inverse frequency)
    num_samples = len(dataset)
    class_weights = {i: num_samples / count for i, count in class_counts.items()}
    print("\nClass weights (inverse frequency):")
    for i, weight in class_weights.items():
        print(f"  {idx_to_class[i]}: {weight:.4f}")
    
    # Assign weight to each sample
    sample_weights = [class_weights[label] for _, label in dataset]
    
    # Visualize weights
    df = pd.DataFrame({
        'Class': [idx_to_class[label] for _, label in dataset],
        'Weight': sample_weights
    })
    
    plt.figure(figsize=(8, 5))
    sns.boxplot(x='Class', y='Weight', data=df)
    plt.title("Sampling Weights Distribution")
    plt.ylabel("Weight")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "sampling_weights.png"))
    plt.close()
    
    return sample_weights, class_counts, class_weights

# Calculate sampling weights
sample_weights, original_counts, class_weights = calculate_sampling_weights(train_dataset)

# Create weighted sampler for balanced training
sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(train_dataset),
    replacement=True
)

# Simulate balanced sampling
def visualize_balanced_sampling(dataset, sampler, num_batches=5, batch_size=100):
    """Visualize effect of balanced sampling"""
    # Create a dataloader with the sampler
    dataloader = DataLoader(dataset, batch_size=batch_size, sampler=sampler)
    
    # Count class distribution in sampled batches
    sampled_counts = {i: 0 for i in range(len(classes))}
    
    for i, (_, labels) in enumerate(dataloader):
        if i >= num_batches:
            break
        
        for label in labels:
            sampled_counts[label.item()] += 1
    
    # Calculate expected counts after balancing
    total_sampled = sum(sampled_counts.values())
    expected_per_class = total_sampled / len(classes)
    
    print("\nClass distribution after balancing (sampled batches):")
    for i, count in sampled_counts.items():
        print(f"  {idx_to_class[i]}: {count} samples ({count/total_sampled*100:.1f}%)")
    
    # Plot comparison
    classes_list = list(classes)
    original_dist = [original_counts[class_to_idx[cls]] for cls in classes_list]
    original_pct = [count/sum(original_dist)*100 for count in original_dist]
    
    sampled_dist = [sampled_counts[class_to_idx[cls]] for cls in classes_list]
    sampled_pct = [count/sum(sampled_dist)*100 for count in sampled_dist]
    
    plt.figure(figsize=(10, 6))
    
    x = np.arange(len(classes_list))
    width = 0.35
    
    plt.bar(x - width/2, original_pct, width, label='Original')
    plt.bar(x + width/2, sampled_pct, width, label='After Sampling')
    
    plt.xlabel('Class')
    plt.ylabel('Percentage (%)')
    plt.title('Class Distribution Before and After Balanced Sampling')
    plt.xticks(x, classes_list)
    
    for i, v in enumerate(original_pct):
        plt.text(i - width/2, v + 1, f"{v:.1f}%", ha='center')
    
    for i, v in enumerate(sampled_pct):
        plt.text(i + width/2, v + 1, f"{v:.1f}%", ha='center')
    
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "balanced_sampling_effect.png"))
    plt.close()

# Visualize the effect of balanced sampling
visualize_balanced_sampling(train_dataset, sampler)


Class distribution before balancing:
  SOBER: 5584 samples
  DRUNK: 2458 samples

Class weights (inverse frequency):
  SOBER: 1.4402
  DRUNK: 3.2718

Class distribution after balancing (sampled batches):
  SOBER: 262 samples (52.4%)
  DRUNK: 238 samples (47.6%)


In [9]:
# ====================== DATA LOADERS ======================

BATCH_SIZE = 100

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=sampler,
    num_workers=4,
    pin_memory=True if torch.cuda.is_available() else False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True if torch.cuda.is_available() else False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True if torch.cuda.is_available() else False
)




In [10]:
# ====================== MODEL ARCHITECTURE ======================

def create_model():
    """Create and configure ResNet-18 model for binary classification"""
    # Load pretrained ResNet-18
    model = models.resnet18(pretrained=True)
    
    print("\nModel Architecture (before modification):")
    print(model)
    
    # Print number of parameters before modification
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\nTotal parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")
    
    # Freeze all layers except final ones
    for name, param in model.named_parameters():
        if "fc" not in name:  # Freeze all layers except the fully connected layer
            param.requires_grad = False
    
    # Modify the final fully connected layer for binary classification
    num_ftrs = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Linear(num_ftrs, 1),
        nn.Sigmoid()
    )
    
    print("\nModel Architecture (after modification):")
    print(model)
    
    # Print number of parameters after modification
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\nTotal parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")
    
    # Visualize model architecture
    visualize_model_architecture(model)
    
    return model

def visualize_model_architecture(model):
    """Visualize model architecture as a table"""
    layers = []
    
    for name, module in model.named_children():
        if isinstance(module, nn.Sequential):
            for sub_name, sub_module in module.named_children():
                layer_name = f"{name}.{sub_name}"
                params = sum(p.numel() for p in sub_module.parameters())
                trainable = sum(p.numel() for p in sub_module.parameters() if p.requires_grad)
                layers.append([layer_name, type(sub_module).__name__, params, trainable])
        else:
            params = sum(p.numel() for p in module.parameters())
            trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)
            layers.append([name, type(module).__name__, params, trainable])
    
    # Create a DataFrame and visualize
    df = pd.DataFrame(layers, columns=['Layer', 'Type', 'Parameters', 'Trainable'])
    df['Frozen'] = df['Parameters'] - df['Trainable']
    df['% Trainable'] = (df['Trainable'] / df['Parameters'] * 100).fillna(0).round(2)
    
    # Plot
    plt.figure(figsize=(12, 8))
    plt.axis('off')
    table = plt.table(
        cellText=df.values,
        colLabels=df.columns,
        cellLoc='center',
        loc='center',
        bbox=[0, 0, 1, 1]
    )
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1, 1.5)
    plt.title("ResNet-18 Architecture for Binary Classification", y=1.08)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "model_architecture.png"), bbox_inches='tight')
    plt.close()

# Create model
model = create_model()
model = model.to(device)


/opt/anaconda3/envs/inebriation-env/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/anaconda3/envs/inebriation-env/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)



Model Architecture (before modification):
ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stat

In [11]:
# ====================== LOSS FUNCTION ======================

class WeightedBinaryCrossEntropyLoss(nn.Module):
    def __init__(self, weight_pos=0.9, weight_neg=0.1):
        super(WeightedBinaryCrossEntropyLoss, self).__init__()
        self.weight_pos = weight_pos
        self.weight_neg = weight_neg
        print("\nWeighted Binary Cross Entropy Loss:")
        print(f"  Positive class (DRUNK) weight: {weight_pos}")
        print(f"  Negative class (SOBER) weight: {weight_neg}")
        
    def forward(self, pred, target):
        target = target.float()
        loss = self.weight_pos * target * torch.log(pred + 1e-7) + \
               self.weight_neg * (1 - target) * torch.log(1 - pred + 1e-7)
        return -torch.mean(loss)

def visualize_loss_function():
    """Visualize the weighted BCE loss function"""
    # Create range of prediction probabilities
    pred_probs = np.linspace(0.001, 0.999, 100)
    
    # Calculate loss for each class
    weight_pos = 0.9
    weight_neg = 0.1
    
    # Loss for positive class (y=1)
    pos_loss = [-weight_pos * np.log(p) for p in pred_probs]
    
    # Loss for negative class (y=0)
    neg_loss = [-weight_neg * np.log(1-p) for p in pred_probs]
    
    # Standard BCE for comparison
    std_pos_loss = [-np.log(p) for p in pred_probs]
    std_neg_loss = [-np.log(1-p) for p in pred_probs]
    
    # Plot
    plt.figure(figsize=(10, 6))
    
    plt.plot(pred_probs, pos_loss, 'b-', label=f'Weighted BCE (y=1, w={weight_pos})')
    plt.plot(pred_probs, neg_loss, 'r-', label=f'Weighted BCE (y=0, w={weight_neg})')
    plt.plot(pred_probs, std_pos_loss, 'b--', alpha=0.5, label='Standard BCE (y=1)')
    plt.plot(pred_probs, std_neg_loss, 'r--', alpha=0.5, label='Standard BCE (y=0)')
    
    plt.title('Weighted Binary Cross Entropy Loss')
    plt.xlabel('Prediction Probability')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "weighted_bce_loss.png"))
    plt.close()

# Visualize loss function
visualize_loss_function()

# Set up loss function
criterion = WeightedBinaryCrossEntropyLoss(weight_pos=0.9, weight_neg=0.1)



Weighted Binary Cross Entropy Loss:
  Positive class (DRUNK) weight: 0.9
  Negative class (SOBER) weight: 0.1


In [12]:
# ====================== OPTIMIZER & SCHEDULER ======================

optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

print("\nOptimizer Configuration:")
print("  Type: Adam")
print("  Initial learning rate: 0.001")
print("  Learning rate scheduler: StepLR (step_size=10, gamma=0.1)")

def visualize_lr_schedule():
    """Visualize the learning rate schedule"""
    # Initialize optimizer and scheduler
    dummy_model = nn.Linear(1, 1)
    dummy_optimizer = optim.Adam(dummy_model.parameters(), lr=0.001)
    dummy_scheduler = optim.lr_scheduler.StepLR(dummy_optimizer, step_size=10, gamma=0.1)
    
    # Track learning rates
    lrs = []
    for epoch in range(30):
        lrs.append(dummy_scheduler.get_last_lr()[0])
        dummy_scheduler.step()
    
    # Plot
    plt.figure(figsize=(10, 5))
    plt.plot(range(1, 31), lrs, 'bo-')
    plt.title('Learning Rate Schedule')
    plt.xlabel('Epoch')
    plt.ylabel('Learning Rate')
    plt.xticks(range(0, 31, 5))
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "learning_rate_schedule.png"))
    plt.close()

# Visualize learning rate schedule
visualize_lr_schedule()


Optimizer Configuration:
  Type: Adam
  Initial learning rate: 0.001
  Learning rate scheduler: StepLR (step_size=10, gamma=0.1)


/opt/anaconda3/envs/inebriation-env/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:182: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn(


In [13]:
# ====================== TRAINING FUNCTIONS ======================

def train_epoch(model, train_loader, criterion, optimizer, epoch):
    """Train the model for one epoch"""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    
    # Use tqdm for progress bar
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}")
    
    for inputs, labels in progress_bar:
        inputs = inputs.to(device)
        labels = labels.to(device)
        
        # Zero the parameter gradients
        optimizer.zero_grad()
        
        # Forward pass
        outputs = model(inputs)
        outputs = outputs.squeeze()
        loss = criterion(outputs, labels)
        
        # Backward pass and optimize
        loss.backward()
        optimizer.step()
        
        # Track statistics
        running_loss += loss.item() * inputs.size(0)
        
        # Convert probabilities to binary predictions
        preds = (outputs >= 0.5).float()
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        
        # Store for metrics calculation
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        
        # Update progress bar
        progress_bar.set_postfix({
            'loss': f"{loss.item():.4f}", 
            'acc': f"{100 * correct / total:.2f}%"
        })
    
    # Calculate epoch metrics
    epoch_loss = running_loss / len(train_loader.dataset)
    epoch_acc = 100 * correct / total
    
    # Additional metrics
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    metrics = {
        'loss': epoch_loss,
        'accuracy': accuracy_score(all_labels, all_preds),
        'precision': precision_score(all_labels, all_preds, zero_division=0),
        'recall': recall_score(all_labels, all_preds, zero_division=0),
        'f1': f1_score(all_labels, all_preds, zero_division=0),
    }
    
    return metrics
def evaluate(model, data_loader, criterion):
    """Evaluate the model on the given data loader"""
    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []  # Store raw probabilities for ROC curve
    running_loss = 0.0
    
    with torch.no_grad():
        for inputs, labels in data_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            
            outputs = model(inputs)
            outputs = outputs.squeeze()
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * inputs.size(0)
            
            # Store raw probabilities
            all_probs.extend(outputs.cpu().numpy())
            
            # Convert probabilities to binary predictions
            preds = (outputs >= 0.5).float()
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    # Calculate metrics
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_probs = np.array(all_probs)
    
    # ROC curve
    fpr, tpr, _ = roc_curve(all_labels, all_probs)
    roc_auc = auc(fpr, tpr)
    
    # Confusion matrix
    cm = confusion_matrix(all_labels, all_preds)
    
    metrics = {
        'loss': running_loss / len(data_loader.dataset),
        'accuracy': accuracy_score(all_labels, all_preds),
        'precision': precision_score(all_labels, all_preds, zero_division=0),
        'recall': recall_score(all_labels, all_preds, zero_division=0),
        'f1': f1_score(all_labels, all_preds, zero_division=0),
        'confusion_matrix': cm,
        'roc': {'fpr': fpr, 'tpr': tpr, 'auc': roc_auc},
        'probabilities': all_probs,
        'true_labels': all_labels
    }
    
    return metrics

NameError: name 'train_model' is not defined